# Part 1：音声と PDF の取り込み
`setup_20261216.sql` を実行してから開始します。AI 関数の実行前に[最新の料金](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-cost)を確認してください。演習では音声と PDF を各1件に限定します。

## 音声と文書をテーブルにする
Snowflake Notebook に SQL セルを追加し、以下を実行します。サンプルの処理結果は `P1_SAMPLE_` テーブルに保存するため、後半用のデータとは混ざりません。音声の文字起こしと、PDFのページ別テキストを確認してください。

```sql
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

CREATE TABLE IF NOT EXISTS P1_SAMPLE_CALL AS
SELECT AI_TRANSCRIBE(TO_FILE('@DATA_STAGE', 'voice_logs/CALL_001.mp3')):text::VARCHAR AS transcript_text;

CREATE TABLE IF NOT EXISTS P1_SAMPLE_PDF AS
SELECT AI_PARSE_DOCUMENT(TO_FILE('@DATA_STAGE', 'documents/quality_report_202611.pdf'),
  {'mode': 'LAYOUT', 'page_split': true}) AS doc;

SELECT transcript_text FROM P1_SAMPLE_CALL;
SELECT page.index + 1 AS page_no, LEFT(page.value:content::VARCHAR, 300) AS page_preview
FROM P1_SAMPLE_PDF, LATERAL FLATTEN(input => doc:pages) page
ORDER BY page_no;
```

文字起こしが空、またはページが0行なら、まず `LIST @DATA_STAGE` でファイルを確認してください。講師から全件スナップショットが配布されている場合は、`checkpoint_20261216.sql` の Part 1 を実行して Part 2 に進めます。